## [Tutorial](https://github.com/yashizhang/esm/tree/main/cookbook/tutorials): How to run minibinder + scFv design fully end-to-end.

In this notebook we will use [Modal](https://modal.com/) to parallelize binder design and synthesize a selection, using the protocol described in the ESMC and ESMFold2 paper titled ["Language Modeling Materializes a World Model of Protein Biology"](https://biohub.ai/papers/esm_protein.pdf).

Biohub used this approach to design minibinders and scFvs against five therapeutically relevant targets — PDGFRB, EGFR, PD-L1, CD45, and CTLA4 — spanning receptor tyrosine kinases, immune checkpoints, and cell-surface phosphatases. Binders exhibit nanomolar affinity, target specificity, and functional activity in laboratory assays.


**You'll need:**
- A target protein sequence (or pick one of the built-in presets)
- A [Modal](https://modal.com/) account and token (free tier works to get started)
- The notebook contains cells launching independent trajectories and a small sweep (N=256). With Modal H100 pricing, this will cost ≈$2 and ≈$150, respectively

**You'll get:** a file of top-ranked designed binder sequences, plus 3D structures of the predicted complexes that you can view in the notebook.

**Workflow:**
1. **Setup** (one-time): install dependencies, get a Modal token, deploy the design app
2. **Try one job**: pick a target and binder type, run a single design end-to-end as a sanity check
3. **Run a sweep**: launch many parallel jobs to produce real candidates
4. **Pick the designs to order**: filter and rank, save the shortlist


> **Why does this notebook need Modal?** Each design job needs a GPU and a few minutes of compute, and a real campaign means launching hundreds of these in parallel. [Modal](https://modal.com/) is a cloud service that lets this notebook spin up remote GPUs on demand, run each job on one, and return results to you. You don't manage any servers or containers yourself. When you "deploy" the design app to Modal (in section 1), you're uploading a Python file that defines the job; Modal then runs that code for you whenever the notebook calls `app.design.spawn(...)`.

### 1. Setup

In [ ]:
# Environment
! pip install esm@git+https://github.com/yashizhang/esm.git@main
! pip install modal py3dmol pyarrow

In [ ]:
# Confirm you have a modal token, or make one
! modal token info  # Check
# ! modal token new  # Create

### Deploy the design app to Modal

The file `binder_design.py` lives in the same folder as this notebook. It defines the GPU job, the model, and the design loop.

You only need to deploy once. Re-run this cell only if the underlying `.py` file changes.

### If you're running on Colab

Colab only pulls the notebook itself when you open it, not the surrounding files from the repo. Run the cell below to grab `binder_design.py` into your Colab workspace. If you're running locally, skip it.

In [ ]:
# Colab only: download binder_design.py into the working directory
! wget -q https://raw.githubusercontent.com/yashizhang/esm/main/cookbook/tutorials/binder_design.py
! ls binder_design.py  

In [ ]:
# Deploy (or redeploy after changing binder_design.py).
# This only needs to be run a single time, unless code in binder_design.py changes.
! modal deploy binder_design.py

### Imports

In [2]:
from itertools import product
from pathlib import Path

import modal
import pandas as pd
import py3Dmol
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

### App setup

`modal.Cls.from_name(...)` grabs a handle to the design app you just deployed, without rerunning anything on Modal yet. Instantiating it gives you `app`, which is what you'll call `app.design.spawn(...)` on to launch design jobs.

`use_scaling_critics=False` is the default. Setting it to `True` adds extra critic models from the paper that improve selection at the cost of more compute per job.

In [ ]:
ESMFold2Design = modal.Cls.from_name("esmfold2-design", "ESMFold2DesignModal")
# Set 'use_scaling_critics' to evaluate with the additional critics.
# Off by default. But cells below were populated with them enabled.
app = ESMFold2Design(use_scaling_critics=False)

## 2. Try one design job

Run a single job end-to-end before launching a sweep. This is a sanity check that everything is wired up and that the target/scaffold combo you've chosen produces a sensible complex.

Pick **one** of the two options below and run only that cell. (They both define a variable called `future`, so running both back-to-back overwrites the first one.)

**Option 1** uses a built-in target and binder scaffold. Available targets: `ctla4`, `egfr`, `pdgfrb`, `pd-l1`, `cd45`. Available binder types: `minibinder`, `trastuzumab_framework_vhvl` (an antibody scaffold). Easiest if your target is one of the built-ins.

**Option 2** takes your own target sequence and binder scaffold. In the binder scaffold, `#` means "design this position" and any amino acid letter means "keep this position fixed." For example:
- `"#" * 60` designs a fully free 60-residue minibinder
- A trastuzumab-style antibody scaffold (shown below) fixes the framework regions and lets the model design the CDR loops 

If you're designing an antibody, pass `is_antibody=True` so the selection step later uses the antibody-appropriate scoring.

Jobs run on Modal in the background. The `dashboard_url` link the cell prints is a clickable link to live progress.

In [4]:
# ---- Option 1: Use presets. ----
# Relies on the registry in modal_binder_design.py::{TARGET_SEQUENCES,BINDER_PROMPT_FACTORIES}, which can be modified.
future = app.design.spawn(target_name="ctla4", binder_name="minibinder")
future.get_dashboard_url()  # A clickable link to Modal dashboard

'https://modal.com/id/fc-01KT1ZA3NQ0JTF2B4HNCR159NM'

In [ ]:
# ---- Option 2: Provide your own sequences. ----
# Our pd-l1 sequence crop.
pdl1_sequence = "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNA"
# A sample of 'trastuzumab_framework_vhvl' template.  From binder_design.py::BINDER_PROMPT_FACTORIES.
trastuzumab_framework_vhvl = "EVQLVESGGGLVQPGGSLRLSCAAS#######YIHWVRQAPGKGLEWVARI#####TRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSR###########WGQGTLVTVSSGGGSGGGSGGGSGGGSDIQMTQSPSSLSASVGDRVTITC###########WYQQKPGKAPKLLIY#######GVPSRFSGSRSGTDFTLTISSLQPEDFATYYC#########FGQGTKVEIK"
future2 = app.design.spawn(
    target_sequence=pdl1_sequence,
    binder_sequence=trastuzumab_framework_vhvl,
    is_antibody=True,
)
future2.get_dashboard_url()  # A clickable link to Modal dashboard

In [ ]:
# ---- Monitor ----
# Tail a function's output here in jupyter
! modal app logs esmfold2-design -f --function-call {future.object_id}

In [ ]:
# ---- Load result ----
best_sequences, trajectory, critic_results = future.get()
print("Best sequences: ", best_sequences)
df = pd.DataFrame(critic_results)
df.drop(columns=["logits", "complex"])

In [54]:
# ---- Visualize ----
protein_complex = (
    df[df.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].iloc[0].complex
)
(
    py3Dmol.view(width=600, height=600)
    .addModel(protein_complex.to_pdb_string(), "pdb")
    .setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})  # pyright: ignore
    .setStyle({"chain": "B"}, {"cartoon": {"color": "cyan"}})  # pyright: ignore
    .addStyle(  # pyright: ignore
        {"not": {"atom": ["N", "C", "O"]}},
        {"stick": {"colorscheme": "default", "radius": 0.2}},
    )
    .center()  # pyright: ignore
    .zoomTo()  # pyright: ignore
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Run a sweep for real designs
For real candidates worth ordering, sweep across many seeds (and optionally multiple targets, binder types, or lengths) and select the best.

Edit `line_sweeps` below to define your campaign. Each key is a sweep axis; the notebook runs one job per combination of values. The default below sweeps 128 seeds across two binder types against PD-L1. 

**Before you click Run on the Launch cell, check the printed shape of the dataframe and confirm it's the number of jobs you intended.** 

In [ ]:
# ---- Config ----
save_dir = Path("sweep")
save_dir.mkdir(exist_ok=True)

# Sweep settings - each key-value pair is an axis of a grid sweep.
line_sweeps = dict(
    target_name=["pd-l1"],
    target_sequence=[None],
    binder_name=["minibinder", "trastuzumab_framework_vhvl"],  # two modalities
    binder_sequence=[None],
    use_scaling_critics=[False],
    seed=list(range(128)),
    batch_size=[1],
)
df = pd.DataFrame(product(*line_sweeps.values()), columns=list(line_sweeps.keys()))
display(df.head(2))
df.shape

,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size
0,pd-l1,None,minibinder,None,False,0,1
1,pd-l1,None,minibinder,None,False,1,1


(256, 7)

In [6]:
# ---- Launch ----
df["call_id"] = [
    app.design.spawn(
        target_name=row.target_name,
        target_sequence=row.target_sequence,
        binder_name=row.binder_name,
        binder_sequence=row.binder_sequence,
        seed=row.seed,
        batch_size=row.batch_size,
    ).object_id
    for row in df.itertuples()
]
df.to_parquet(save_dir / "manifest.parquet", index=False)
print(
    f"Spawned {len(df)} jobs. It is safe to close the notebook."
    "The next cell will resume from call_id's, saved by Modal for up to 7 days."
)

Spawned 256 jobs. It is safe to close the notebook.The next cell will resume from call_id's, saved by Modal for up to 7 days.


### Coming back later (important)

Your jobs are now running on Modal's GPUs. **You do not need to keep this notebook open.** Modal continues running the jobs on its own and holds the results for up to 7 days.

**If you're running on Colab:** the runtime is wiped when you disconnect. To resume a sweep on Colab, mount Google Drive **before** running the Launch cell and point `save_dir` at a Drive path, e.g.:

\`\`\`python
from google.colab import drive
drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/binder_sweep')
\`\`\`

When you reopen the notebook later, you'll need to re-install the dependencies, re-authenticate Modal (`modal token new`), re-mount Drive, then resume from the Monitor cell.

In [34]:
# ---- Monitor ----
from tqdm.contrib.concurrent import thread_map

df = pd.read_parquet(save_dir / "manifest.parquet")
df["future"] = thread_map(modal.FunctionCall.from_id, df.call_id.values)
df["status"] = thread_map(lambda f: f.get_call_graph()[0].status.name, df.future.values)
print("First task url: ", df.at[0, "future"].get_dashboard_url())  # pyright: ignore
df.status.value_counts()

  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

First task url:  https://modal.com/id/fc-01KT1ZA3SPBK7KPYDKFKERVWGY


status
SUCCESS    256
Name: count, dtype: int64

The Collect cell waits for all jobs that succeeded and unpacks their results. Jobs that failed or are still running are skipped, so you can re-run Monitor + Collect periodically as more jobs finish.

In [48]:
# ---- Collect ----
df_success = df[df.status.eq("SUCCESS")].copy()
df_success["result"] = thread_map(
    lambda x: x.get(), df_success.future.values, max_workers=32, chunksize=32
)  # Blocks until all jobs are complete.
df_success["result_df"] = [pd.DataFrame(r[2]) for r in tqdm(df_success.result)]  # pyright: ignore

  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

## 4. Pick the designs to order

This is your final shortlist. The cell below:

1. Combines results from all successful jobs into one dataframe.
2. Filters minibinders to isoelectric point under 6 (helps with solubility and expression). Antibodies pass through unfiltered.
3. Scores each unique designed sequence by averaging `iptm` (interface predicted TM-score, higher is better) and an `iptm_proxy` term across its trajectories.
4. Returns the top 84 designs per (target, binder type), saved to `selection.parquet` inside your `save_dir`.

84 is a plate-friendly number for ordering and screening. The cutoff and the isoelectric-point filter are currently hardcoded inside the cell, so to change them, edit the values directly in the function.


In [49]:
# ---- Select ----

# Join all result_df's, with other fields in df broadcasted as metadata.
df_result = pd.concat(
    [
        row.result_df.assign(**row.drop(["result", "result_df"]).to_dict())  # pyright: ignore
        for _, row in df_success.iterrows()
    ],
    ignore_index=True,
    axis=0,
)

# Filter minibinder designs with isoelectric point >= 6.
df_result["binder_sequence"] = df_result.designed_sequence.str.split(r"\|").str[1]
df_result["isoelectric_point"] = [
    ProteinAnalysis(seq).isoelectric_point()
    for seq in tqdm(df_result.binder_sequence.values)
]
# Isoelectric point filter
df_filter = df_result[df_result.is_antibody | df_result.isoelectric_point.lt(6)]


# Select the top 84 designs from each (target, binder) combination
SCALING_CHECKPOINT_SUBSTRING = "ESMFold2-Experimental-Fast-base"


def select(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    is_scaling = df.critic_name.str.contains(
        SCALING_CHECKPOINT_SUBSTRING, regex=False, na=False
    )
    iptm_proxy = df.distogram_iptm_proxy.where(
        ~df.is_antibody, df.cdr_distogram_iptm_proxy
    )

    df["iptm_score"] = df.iptm.where(~is_scaling)
    df["iptm_proxy_score"] = iptm_proxy.where(is_scaling)
    scores = df.groupby("designed_sequence", as_index=False).agg(
        iptm_score=("iptm_score", "mean"), iptm_proxy_score=("iptm_proxy_score", "mean")
    )
    scores["selection_score"] = 0.5 * scores.iptm_score.fillna(
        0
    ) + 0.5 * scores.iptm_proxy_score.fillna(0)

    return scores.nlargest(min(len(scores), 84), "selection_score")


df_select = df_filter.groupby(["target_name", "binder_name"]).apply(
    select, include_groups=False
)
df_select.to_parquet(save_dir / "selection.parquet", index=False)
df_select

  0%|          | 0/4864 [00:00<?, ?it/s]

designed_sequence  \
target_name binder_name                                                                         
pd-l1       minibinder                 70   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       13   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       78   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       75   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       45   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
...                                                                                       ...   
            trastuzumab_framework_vhvl 2    AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       107  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       46   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       68   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       50   AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   

                                            iptm_score  iptm_proxy_score  \
target_name binder_name                                                    
pd-l1       minibinder                 70     0.967982          0.944397   
                                       13     0.964470          0.926216   
                                       78     0.961920          0.918186   
                                       75     0.957516          0.916083   
                                       45     0.964665          0.908466   
...                                                ...               ...   
            trastuzumab_framework_vhvl 2      0.921884          0.725663   
                                       107    0.920620          0.726391   
                                       46     0.879064          0.767323   
                                       68     0.921196          0.724139   
                                       50     0.894414          0.747255   

                                            selection_score  
target_name binder_name                                      
pd-l1       minibinder                 70          0.956190  
                                       13          0.945343  
                                       78          0.940053  
                                       75          0.936799  
                                       45          0.936566  
...                                                     ...  
            trastuzumab_framework_vhvl 2           0.823773  
                                       107         0.823506  
                                       46          0.823194  
                                       68          0.822668  
                                       50          0.820834  

[168 rows x 4 columns]

In [ ]:
df_result[df_result.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].drop(
    columns=["complex", "logits"]
)

In [ ]:
df_select

## Appendix

### Modal Primer

- **info: ephemeral vs deployment**  
  Ephemeral = temporary app from `modal run` or `app.run()`, stopped when the client exits. Deployment = persistent named app from `modal deploy`, reused and observable across runs. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **info: dashboard**  
  Generic dashboard/apps page: [https://modal.com/apps](https://modal.com/apps). Modal also prints app/deployment links during runs/deploys. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **cli: ephemeral run**  
  ```bash
  modal run path/to/app.py
  ```

- **cli: deploy/redeploy**  
  ```bash
  modal deploy path/to/app.py
  ```
  Running this on an existing app name redeploys a new version. ([modal.com](https://modal.com/docs/reference/cli/deploy?utm_source=openai))

- **local: ephemeral from Python**  
  ```python
  with modal.enable_output():
      with modal_app.run():
          result = local_modal_obj.remote(...)
  ```

- **local: call a deployment**  
  ```python
  Cls = modal.Cls.from_name("app-name", "ClassName")
  obj = Cls(...)
  result = obj.method.remote(...)
  ```
  `Cls.from_name` references a class from a deployed app lazily. ([modal.com](https://modal.com/docs/reference/modal.Cls?utm_source=openai))